# 0️⃣2️⃣ - How to Connect to My RADKit Service

![Persona](https://img.shields.io/badge/%F0%9F%91%A4%20Persona-%F0%9F%A4%96%20Network%20Automation%20Developer-lightgrey) ![Difficulty](https://img.shields.io/badge/%F0%9F%9A%A6%20Difficulty-Beginner-green) ![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-yellow?logo=python&logoColor=white)

> In this playbook we explore the two ways to connect your script to a RADKit service: **via the cloud** and **via a direct connection**.

---

### 📋 What you'll learn

| # | Topic |
|---|-------|
| 1 | What the `Service` object is and why it matters |
| 2 | How to authenticate and connect **through the Cisco cloud** (SSO login) |
| 3 | How to connect **directly** to a RADKit server over the network (no cloud needed) |
| 4 | When to use each connection method |

### 🛠️ Before You Begin

If you haven't set up your environment yet, follow the instructions in [SETUP.md](../SETUP.md) before running any cells.

> **Prerequisite:** Complete [playbook 01](01-authentication-connection.ipynb) first, as it covers authentication concepts used throughout this notebook.

---

### 🤖 The `Service` Object

The `Service` object is your main entry point to everything inside a RADKit service. Once you have a `Service` instance, you can:

- Browse and query the **device inventory**
- Send commands to devices
- Run scripts, filters, and queries

Think of it as a **live handle to your RADKit deployment**, with all subsequent API calls flowing through it.

There are two ways to obtain a `Service` object, each suited to a different network scenario:

---

## ☁️ Method 1: Connect via the Cisco Cloud (SSO Login)

**Best for:** Most users. Authentication goes through Cisco's cloud using your CCO (Cisco.com) account, so no network-level access to the RADKit server is required from your machine.

**How it works:**
1. A `Client` is created and you log in with your CCO credentials via SSO (a browser window will open).
2. Once authenticated, the client reaches the RADKit service through the cloud using its **Service ID**.

> 💡 Your **Service ID** is displayed at the top of your RADKit server's web UI. See [SETUP.md](../SETUP.md) for a screenshot.

**What you need:**
- Your **CCO user ID** (e.g. `jsmith@cisco.com`)
- The **Service ID** of the RADKit service you want to connect to

In [2]:
from radkit_client import Client
from radkit_client.sync import ClientStatus, ServiceStatus # Nice enums to get the status of the client and service

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    print("✅ Authentication successful!") if client.status == ClientStatus.CONNECTED else print("🔥 Connection failed ...")
    
    service = client.service_cloud(service_id).wait()
    print(f"✅ Connection successful to service {service.service_id}!") if service.status == ServiceStatus.READY else print(f"🔥 Connection to service {service.service_id} failed ...")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.
✅ Authentication successful!
✅ Connection successful to service uhfm-4hm2-b5bx!


---

## ↔️ Method 2: Connect Directly (No Cloud)

**Best for:** Air-gapped environments, CI/CD pipelines, or any situation where outbound internet access is restricted or undesirable.

**How it works:** The client connects directly to the RADKit server over your local network or VPN, bypassing the Cisco cloud entirely.

**Requirements:**

| Requirement | Details |
|---|---|
| Network reachability | The RADKit server must be accessible from the machine running this script |
| Direct RPC enabled | Check/enable on the server UI: `Settings > service.connectivity.port_direct_rpc` (default port: **8181**) |
| E2EE validation token | Found in the server UI under `Remote Users > <your user> > E2EE validation token` |

**What you need:**
- Your **CCO user ID**
- Your **E2EE validation token** (acts as a password for direct connections)
- The **server address** (hostname or IP)
- The **RPC port** (default: `8181`)

In [ ]:
from radkit_client import Client
from radkit_common.rpc.client_transports.verify import RPCVerificationError # Specific exception for failed RPC verification
import getpass

user_id = input("👤 Enter your CCO user id: ")
e2ee_validation_token = getpass.getpass("🔑 Enter your E2EE validation token: ")
server_address = input("🌐 Enter the server address (e.g., https://my-radkit-server.com): ")
rpc_port = input("🔌 Enter the RPC port of your server (default is 8181): ")

with Client.create() as client:
    service = client.service_direct(
        username=user_id,
        host=server_address,
        port=int(rpc_port) if rpc_port else 8181,
        password=e2ee_validation_token
    )
    try:
        service.wait()
        print(f"✅ Connection successful to service!")
    except RPCVerificationError as e:
        print(f"🔥 Connection to service failed: {e}")
    except Exception as e:
        print(f"🔥 An unexpected error occurred while connecting to service: {e}")

✅ Connection successful to service!


---

## 🔀 Which Method Should I Use?

| | ☁️ Cloud (SSO) | ↔️ Direct |
|---|---|---|
| **Network required** | Internet only | LAN/VPN to RADKit server |
| **Authentication** | CCO credentials via browser | E2EE validation token |
| **Setup effort** | Minimal | Requires enabling direct RPC on server |
| **Best for** | General use, remote access | Air-gapped environments, CI/CD pipelines |